similar customer reviews

In [2]:
reviews = [
    "The product quality is excellent and delivery was fast!",
    "Amazing service! Quick shipping and great customer support.",
    "The item arrived late, but the quality was good.",
    "Not satisfied. The product was damaged on arrival.",
    "Fast delivery but poor packaging.",
    "Superb experience! Will buy again."
]


In [4]:
from sentence_transformers import SentenceTransformer  

model = SentenceTransformer("all-MiniLM-L6-v2")  
review_vectors = model.encode(reviews)  


In [5]:
from qdrant_client import QdrantClient  
from qdrant_client.models import Distance, VectorParams  

client = QdrantClient("localhost", port=6333)  

client.recreate_collection(  
    collection_name="reviews",  
    vectors_config=VectorParams(size=384, distance=Distance.COSINE)  
)


C:\Users\admin\AppData\Local\Temp\ipykernel_11224\989966152.py:6: DeprecationWarning: `recreate_collection` method is deprecated and will be removed in the future. Use `collection_exists` to check collection existence and `create_collection` instead.
  client.recreate_collection(


True

In [6]:
client.upsert(  
    collection_name="reviews",  
    points=[  
        {"id": i, "vector": review_vectors[i].tolist(), "payload": {"text": reviews[i]}}  
        for i in range(len(reviews))  
    ]  
)


UpdateResult(operation_id=0, status=<UpdateStatus.COMPLETED: 'completed'>)

In [10]:
query = "The product was great but arrived late."  
query_vector = model.encode([query]).tolist()[0]  

results = client.search(  
    collection_name="reviews",  
    query_vector=query_vector,  
    limit=3  
)  

for res in results:  
    print(f"Similar Review: {res.payload['text']} (Score: {res.score:.4f})")  


Similar Review: The item arrived late, but the quality was good. (Score: 0.7241)
Similar Review: The product quality is excellent and delivery was fast! (Score: 0.6853)
Similar Review: Not satisfied. The product was damaged on arrival. (Score: 0.5493)


C:\Users\admin\AppData\Local\Temp\ipykernel_11224\3800012282.py:4: DeprecationWarning: `search` method is deprecated and will be removed in the future. Use `query_points` instead.
  results = client.search(


Performing Basic Statistics on Review Data Using Qdrant

In [13]:
reviews = [
    "The product quality is excellent and delivery was fast!",  # Positive
    "Amazing service! Quick shipping and great customer support.",  # Positive
    "The item arrived late, but the quality was good.",  # Neutral
    "Not satisfied. The product was damaged on arrival.",  # Negative
    "Fast delivery but poor packaging.",  # Neutral
    "Superb experience! Will buy again."  # Positive
]


In [15]:
from sentence_transformers import SentenceTransformer  

model = SentenceTransformer("all-MiniLM-L6-v2")  
review_vectors = model.encode(reviews)  


In [17]:
from qdrant_client import QdrantClient  
from qdrant_client.models import Distance, VectorParams  

client = QdrantClient("localhost", port=6333)  

client.recreate_collection(  
    collection_name="reviews",  
    vectors_config=VectorParams(size=384, distance=Distance.COSINE)  
)

client.upsert(  
    collection_name="reviews",  
    points=[  
        {"id": i, "vector": review_vectors[i].tolist(), "payload": {"text": reviews[i]}}  
        for i in range(len(reviews))  
    ]  
)


C:\Users\admin\AppData\Local\Temp\ipykernel_11224\984478881.py:6: DeprecationWarning: `recreate_collection` method is deprecated and will be removed in the future. Use `collection_exists` to check collection existence and `create_collection` instead.
  client.recreate_collection(


UpdateResult(operation_id=0, status=<UpdateStatus.COMPLETED: 'completed'>)

In [19]:
count = client.count(collection_name="reviews")
print(f"Total Reviews: {count.count}")


Total Reviews: 6


In [35]:
query = "Good quality but slow delivery."  
query_vector = model.encode([query]).tolist()[0]  

results = client.search(  
    collection_name="reviews",  
    query_vector=query_vector,  
    limit=3  
)  

for res in results:  
    print(f"Similar Review: {res.payload['text']} (Score: {res.score:.4f})")  


Similar Review: The product quality is excellent and delivery was fast! (Score: 0.7925)
Similar Review: Fast delivery but poor packaging. (Score: 0.7915)
Similar Review: The item arrived late, but the quality was good. (Score: 0.6482)


C:\Users\admin\AppData\Local\Temp\ipykernel_11224\3043316876.py:4: DeprecationWarning: `search` method is deprecated and will be removed in the future. Use `query_points` instead.
  results = client.search(


In [39]:
sentiments = ["positive", "positive", "neutral", "negative", "neutral", "positive"]
sentiment_counts = {s: sentiments.count(s) for s in set(sentiments)}

print(f"Sentiment Distribution: {sentiment_counts}")


Sentiment Distribution: {'neutral': 2, 'positive': 3, 'negative': 1}


Advanced Review Analysis: Clustering & Visualization with Qdrant 

In [222]:
from sentence_transformers import SentenceTransformer
from qdrant_client.models import PointStruct

# Load embedding model
model = SentenceTransformer("all-MiniLM-L6-v2")

# Sample reviews
texts = ["Great product!", "Not worth the price.", "Loved the experience!", 
         "Terrible quality.", "Highly recommended!", "Very disappointing."]

# Generate embeddings
vectors = model.encode(texts).tolist()

# Insert reviews with vectors
points = [
    PointStruct(id=i + 1, vector=vectors[i], payload={"text": texts[i]})
    for i in range(len(texts))
]

client.upsert(collection_name="reviews1", points=points)
print("✅ Data inserted successfully!")


✅ Data inserted successfully!


In [224]:
all_records, _ = client.scroll(collection_name="reviews1", limit=10)
for record in all_records:
    print(f"ID: {record.id}, Vector: {record.vector is not None}, Payload: {record.payload}")


ID: 1, Vector: False, Payload: {'text': 'Great product!'}
ID: 2, Vector: False, Payload: {'text': 'Not worth the price.'}
ID: 3, Vector: False, Payload: {'text': 'Loved the experience!'}
ID: 4, Vector: False, Payload: {'text': 'Terrible quality.'}
ID: 5, Vector: False, Payload: {'text': 'Highly recommended!'}
ID: 6, Vector: False, Payload: {'text': 'Very disappointing.'}


In [226]:
from sentence_transformers import SentenceTransformer

# Load embedding model
model = SentenceTransformer("all-MiniLM-L6-v2")

# Sample texts
texts = ["Great product!", "Not worth the price.", "Loved the experience!",
         "Terrible quality.", "Highly recommended!", "Very disappointing."]

# Generate embeddings
vectors = model.encode(texts)

# Print shape
print("Shape of generated vectors:", vectors.shape)
print("First vector sample:", vectors[0])


Shape of generated vectors: (6, 384)
First vector sample: [-7.35023171e-02  9.33607295e-02 -4.71811654e-04 -3.71412747e-02
  6.28743246e-02  2.81565320e-02  4.84687276e-02 -7.29220035e-03
 -2.21003275e-02 -5.26671894e-02  3.95704173e-02  1.44291729e-01
  5.40885739e-02  5.61106429e-02 -8.27428326e-03  4.70017157e-02
 -9.04561877e-02  4.35278416e-02  4.38528843e-02 -3.97158861e-02
 -4.90863584e-02 -6.84470236e-02  3.92240696e-02  2.73028426e-02
 -2.95478981e-02  1.82082355e-02 -1.73152369e-02 -3.01048579e-03
  4.21828218e-02 -1.03022479e-01 -4.83286940e-02  1.78528912e-02
 -6.53382231e-05 -2.90111937e-02 -1.48928212e-02 -4.09579873e-02
  2.89271623e-02 -6.88656569e-02 -4.95055728e-02  1.84821133e-02
  1.62871424e-02 -1.26823904e-02 -1.34520810e-02  1.88252646e-02
  5.29956026e-03  7.08957240e-02  3.03423479e-02  3.24798897e-02
  2.64840666e-02  3.72343324e-02 -5.53786084e-02 -2.62467396e-02
 -3.65869068e-02  1.03191296e-02 -9.91416723e-02  9.17618978e-04
  2.05050744e-02 -6.21679761e-02

In [228]:
from qdrant_client.models import PointStruct

# Convert vectors to list (Qdrant requires Python lists)
vectors_list = vectors.tolist()

# Prepare points
points = [
    PointStruct(id=i + 1, vector=vectors_list[i], payload={"text": texts[i]})
    for i in range(len(texts))
]

# Insert into Qdrant
client.upsert(collection_name="reviews1", points=points)

print("✅ Data inserted successfully!")


✅ Data inserted successfully!


In [230]:
all_records, _ = client.scroll(collection_name="reviews1", limit=10)
for record in all_records:
    print(f"ID: {record.id}, Vector Exists: {record.vector is not None}, Payload: {record.payload}")


ID: 1, Vector Exists: False, Payload: {'text': 'Great product!'}
ID: 2, Vector Exists: False, Payload: {'text': 'Not worth the price.'}
ID: 3, Vector Exists: False, Payload: {'text': 'Loved the experience!'}
ID: 4, Vector Exists: False, Payload: {'text': 'Terrible quality.'}
ID: 5, Vector Exists: False, Payload: {'text': 'Highly recommended!'}
ID: 6, Vector Exists: False, Payload: {'text': 'Very disappointing.'}


In [232]:
collection_info = client.get_collection("reviews1")
print(collection_info)


status=<CollectionStatus.GREEN: 'green'> optimizer_status=<OptimizersStatusOneOf.OK: 'ok'> vectors_count=None indexed_vectors_count=0 points_count=6 segments_count=4 config=CollectionConfig(params=CollectionParams(vectors=VectorParams(size=384, distance=<Distance.COSINE: 'Cosine'>, hnsw_config=None, quantization_config=None, on_disk=None, datatype=None, multivector_config=None), shard_number=1, sharding_method=None, replication_factor=1, write_consistency_factor=1, read_fan_out_factor=None, on_disk_payload=True, sparse_vectors=None), hnsw_config=HnswConfig(m=16, ef_construct=100, full_scan_threshold=10000, max_indexing_threads=0, on_disk=False, payload_m=None), optimizer_config=OptimizersConfig(deleted_threshold=0.2, vacuum_min_vector_number=1000, default_segment_number=0, max_segment_size=None, memmap_threshold=None, indexing_threshold=20000, flush_interval_sec=5, max_optimization_threads=None), wal_config=WalConfig(wal_capacity_mb=32, wal_segments_ahead=0), quantization_config=None, 

In [234]:
from qdrant_client.models import CollectionConfig, VectorParams, Distance

client.recreate_collection(
    collection_name="reviews1",
    vectors_config=VectorParams(size=384, distance=Distance.COSINE)
)
print("✅ Collection recreated with vector support!")


C:\Users\admin\AppData\Local\Temp\ipykernel_11224\973990374.py:3: DeprecationWarning: `recreate_collection` method is deprecated and will be removed in the future. Use `collection_exists` to check collection existence and `create_collection` instead.
  client.recreate_collection(


✅ Collection recreated with vector support!


In [236]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")
texts = ["Great product!", "Not worth the price.", "Loved the experience!",
         "Terrible quality.", "Highly recommended!", "Very disappointing."]

vectors = model.encode(texts)
print("Vector shape:", vectors.shape)
print("First vector:", vectors[0])


Vector shape: (6, 384)
First vector: [-7.35023171e-02  9.33607295e-02 -4.71811654e-04 -3.71412747e-02
  6.28743246e-02  2.81565320e-02  4.84687276e-02 -7.29220035e-03
 -2.21003275e-02 -5.26671894e-02  3.95704173e-02  1.44291729e-01
  5.40885739e-02  5.61106429e-02 -8.27428326e-03  4.70017157e-02
 -9.04561877e-02  4.35278416e-02  4.38528843e-02 -3.97158861e-02
 -4.90863584e-02 -6.84470236e-02  3.92240696e-02  2.73028426e-02
 -2.95478981e-02  1.82082355e-02 -1.73152369e-02 -3.01048579e-03
  4.21828218e-02 -1.03022479e-01 -4.83286940e-02  1.78528912e-02
 -6.53382231e-05 -2.90111937e-02 -1.48928212e-02 -4.09579873e-02
  2.89271623e-02 -6.88656569e-02 -4.95055728e-02  1.84821133e-02
  1.62871424e-02 -1.26823904e-02 -1.34520810e-02  1.88252646e-02
  5.29956026e-03  7.08957240e-02  3.03423479e-02  3.24798897e-02
  2.64840666e-02  3.72343324e-02 -5.53786084e-02 -2.62467396e-02
 -3.65869068e-02  1.03191296e-02 -9.91416723e-02  9.17618978e-04
  2.05050744e-02 -6.21679761e-02  2.02453770e-02 -1.0

In [238]:
from qdrant_client.models import PointStruct

vectors_list = vectors.tolist()

points = [
    PointStruct(id=i+1, vector=vectors_list[i], payload={"text": texts[i]})
    for i in range(len(texts))
]

client.upsert(collection_name="reviews1", points=points)
print("✅ Data inserted successfully!")


✅ Data inserted successfully!


In [240]:
all_records, _ = client.scroll(collection_name="reviews1", limit=10)
for record in all_records:
    print(f"ID: {record.id}, Vector Exists: {record.vector is not None}, Payload: {record.payload}")


ID: 1, Vector Exists: False, Payload: {'text': 'Great product!'}
ID: 2, Vector Exists: False, Payload: {'text': 'Not worth the price.'}
ID: 3, Vector Exists: False, Payload: {'text': 'Loved the experience!'}
ID: 4, Vector Exists: False, Payload: {'text': 'Terrible quality.'}
ID: 5, Vector Exists: False, Payload: {'text': 'Highly recommended!'}
ID: 6, Vector Exists: False, Payload: {'text': 'Very disappointing.'}


In [242]:
from qdrant_client import QdrantClient
from qdrant_client.models import CollectionConfig, VectorParams, Distance

# Connect to Qdrant
client = QdrantClient("localhost", port=6333)  # Change if using cloud

# Recreate collection with vector support
client.recreate_collection(
    collection_name="reviews1",
    vectors_config=VectorParams(size=384, distance=Distance.COSINE)
)

print("✅ Collection is ready to store vectors!")


C:\Users\admin\AppData\Local\Temp\ipykernel_11224\2467164496.py:8: DeprecationWarning: `recreate_collection` method is deprecated and will be removed in the future. Use `collection_exists` to check collection existence and `create_collection` instead.
  client.recreate_collection(


✅ Collection is ready to store vectors!


In [244]:
from sentence_transformers import SentenceTransformer

# Load model
model = SentenceTransformer("all-MiniLM-L6-v2")

# Text reviews
texts = ["Great product!", "Not worth the price.", "Loved the experience!",
         "Terrible quality.", "Highly recommended!", "Very disappointing."]

# Convert text to vectors
vectors = model.encode(texts).tolist()  # Convert NumPy array to list

print("✅ Vectors generated successfully!")


✅ Vectors generated successfully!


In [246]:
from qdrant_client.models import PointStruct

# Create points with vectors
points = [
    PointStruct(id=i+1, vector=vectors[i], payload={"text": texts[i]})
    for i in range(len(texts))
]

# Insert into Qdrant
client.upsert(collection_name="reviews1", points=points)

print("✅ Data inserted successfully!")


✅ Data inserted successfully!


In [248]:
all_records, _ = client.scroll(collection_name="reviews1", limit=10)

for record in all_records:
    print(f"ID: {record.id}, Vector Exists: {record.vector is not None}, Payload: {record.payload}")


ID: 1, Vector Exists: False, Payload: {'text': 'Great product!'}
ID: 2, Vector Exists: False, Payload: {'text': 'Not worth the price.'}
ID: 3, Vector Exists: False, Payload: {'text': 'Loved the experience!'}
ID: 4, Vector Exists: False, Payload: {'text': 'Terrible quality.'}
ID: 5, Vector Exists: False, Payload: {'text': 'Highly recommended!'}
ID: 6, Vector Exists: False, Payload: {'text': 'Very disappointing.'}


In [258]:
import numpy as np  

# Retrieve all stored vectors
all_records, _ = client.scroll(collection_name="reviews1", limit=10)

# Extract vectors and labels
all_vectors = np.array([record.vector for record in all_records])
labels = [record.payload["text"] for record in all_records]

print(f"✅ Retrieved {len(all_vectors)} vectors for clustering.")


✅ Retrieved 6 vectors for clustering.
